## TensorFlow Lite Model Script

Purpose:
    Comprehensive  analysis of a TensorFlow Lite (.tflite) model

Usage:<br>
    1. Place .tflite file in the same directory<br>
    2. Change MODEL_PATH below<br>
    3. Run<br>

Recommended:
    pip install tensorflow matplotlib pandas numpy seaborn

### Imports and Scientific style

In [45]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

# Scientific plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

#### Configuration

In [46]:
MODEL_PATH = "hand_landmark_quant.tflite"

#### Helper Functions

In [47]:
def bytes_to_human_readable(size_bytes):
    """
    Convert bytes to human-readable format safely.
    Handles negative values correctly.
    """

    if size_bytes == 0:
        return "0 B"

    # Preserve sign
    sign = "-" if size_bytes < 0 else ""

    size_bytes = abs(size_bytes)

    size_names = ("B", "KB", "MB", "GB", "TB")

    i = int(np.floor(np.log(size_bytes) / np.log(1024)))
    i = min(i, len(size_names) - 1)

    p = np.power(1024, i)

    s = round(size_bytes / p, 2)

    return f"{sign}{s} {size_names[i]}"

def print_section(title):
    """
    Pretty scientific section printing.
    """
    line = "=" * 80
    print(f"\n{line}")
    print(title)
    print(line)


def detect_quantization(tensor_details):
    """
    Detect quantization type from tensor metadata.
    """

    quantized_tensors = []

    for tensor in tensor_details:
        dtype = tensor["dtype"]

        if dtype in [np.int8, np.uint8, np.int16]:
            quantized_tensors.append(tensor)

    if not quantized_tensors:
        return "Float Model (No Quantization)"

    dtypes = set([str(t["dtype"]) for t in quantized_tensors])

    if np.int8 in [t["dtype"] for t in quantized_tensors]:
        return "INT8 Quantization"

    if np.uint8 in [t["dtype"] for t in quantized_tensors]:
        return "UINT8 Quantization"

    if np.float16 in [t["dtype"] for t in quantized_tensors]:
        return "FLOAT16 Quantization"

    return f"Mixed Quantization ({dtypes})"


def tensor_memory_size(shape, dtype):
    """
    Estimate tensor memory footprint.
    """

    try:
        element_count = np.prod(shape)
        dtype_size = np.dtype(dtype).itemsize
        return int(element_count * dtype_size)

    except:
        return 0

#### Load Model

In [48]:
print_section("MODEL LOADING")

print(f"TensorFlow Version: {tf.__version__}")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

model_size = os.path.getsize(MODEL_PATH)

print(f"Model Path: {MODEL_PATH}")
print(f"Model Size: {bytes_to_human_readable(model_size)}")

# Load interpreter
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

print("\nInterpreter successfully initialized.")


MODEL LOADING
TensorFlow Version: 2.19.1
Model Path: hand_landmark_quant.tflite
Model Size: 3.12 MB

Interpreter successfully initialized.


/home/oliver/miniconda3/envs/ml-pipeline/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


#### Basic Model Information

In [49]:
print_section("BASIC MODEL INFORMATION")

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
tensor_details = interpreter.get_tensor_details()

print(f"Number of Input Tensors : {len(input_details)}")
print(f"Number of Output Tensors: {len(output_details)}")
print(f"Total Tensor Count      : {len(tensor_details)}")


BASIC MODEL INFORMATION
Number of Input Tensors : 1
Number of Output Tensors: 4
Total Tensor Count      : 172


#### Quantization Analysis

In [50]:
print_section("QUANTIZATION ANALYSIS")

quantization_type = detect_quantization(tensor_details)

print(f"Detected Quantization Type: {quantization_type}")

print("\nInterpretation:")
if "Float" in quantization_type:
    print("""
The model appears to use floating-point precision throughout.
This usually yields higher numerical precision but increases
memory usage and inference latency on embedded hardware.
""")

elif "INT8" in quantization_type:
    print("""
The model uses INT8 quantization.
This is highly optimized for edge deployment and significantly
reduces memory footprint and inference latency.
""")

elif "UINT8" in quantization_type:
    print("""
The model uses UINT8 quantization.
This is common in older TensorFlow Lite pipelines and microcontroller
deployments.
""")


QUANTIZATION ANALYSIS
Detected Quantization Type: INT8 Quantization

Interpretation:

The model uses INT8 quantization.
This is highly optimized for edge deployment and significantly
reduces memory footprint and inference latency.



#### Input Vector Analysis

In [51]:
print_section("INPUT VECTOR ANALYSIS")

for idx, inp in enumerate(input_details):

    print(f"\nINPUT TENSOR #{idx}")

    print(f"Name              : {inp['name']}")
    print(f"Shape             : {inp['shape']}")
    print(f"Data Type         : {inp['dtype']}")
    print(f"Quantization      : {inp['quantization']}")
    print(f"Quantization Params: {inp['quantization_parameters']}")

    input_memory = tensor_memory_size(inp['shape'], inp['dtype'])

    print(f"Estimated Memory  : {bytes_to_human_readable(input_memory)}")


INPUT VECTOR ANALYSIS

INPUT TENSOR #0
Name              : new_input_tensor
Shape             : [  1 224 224   3]
Data Type         : <class 'numpy.uint8'>
Quantization      : (0.003921568393707275, 0)
Quantization Params: {'scales': array([0.00392157], dtype=float32), 'zero_points': array([0], dtype=int32), 'quantized_dimension': 0}
Estimated Memory  : 147.0 KB


#### Output Vector Analysis

In [52]:
print_section("OUTPUT VECTOR ANALYSIS")

for idx, out in enumerate(output_details):

    print(f"\nOUTPUT TENSOR #{idx}")

    print(f"Name              : {out['name']}")
    print(f"Shape             : {out['shape']}")
    print(f"Data Type         : {out['dtype']}")
    print(f"Quantization      : {out['quantization']}")
    print(f"Quantization Params: {out['quantization_parameters']}")

    output_memory = tensor_memory_size(out['shape'], out['dtype'])

    print(f"Estimated Memory  : {bytes_to_human_readable(output_memory)}")


OUTPUT VECTOR ANALYSIS

OUTPUT TENSOR #0
Name              : Identity_2:0
Shape             : [1 1]
Data Type         : <class 'numpy.float32'>
Quantization      : (0.0, 0)
Quantization Params: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}
Estimated Memory  : 4.0 B

OUTPUT TENSOR #1
Name              : Identity_3:0
Shape             : [ 1 63]
Data Type         : <class 'numpy.float32'>
Quantization      : (0.0, 0)
Quantization Params: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}
Estimated Memory  : 252.0 B

OUTPUT TENSOR #2
Name              : Identity_1:0
Shape             : [1 1]
Data Type         : <class 'numpy.float32'>
Quantization      : (0.0, 0)
Quantization Params: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}
Estimated Memory  : 4.0 B

OUTPUT TENSOR #3
Name              : Identity:0
Shape             : [ 1 63

#### Tensor Overview Table

In [53]:
print_section("TENSOR OVERVIEW TABLE")

tensor_rows = []

for tensor in tensor_details:

    memory = tensor_memory_size(
        tensor["shape"],
        tensor["dtype"]
    )

    tensor_rows.append({
        "Name": tensor["name"],
        "Shape": str(tensor["shape"]),
        "DType": str(tensor["dtype"]),
        "Quantization": str(tensor["quantization"]),
        "MemoryBytes": memory
    })

tensor_df = pd.DataFrame(tensor_rows)

print(tensor_df.head(20))


TENSOR OVERVIEW TABLE
                                                 Name              Shape  \
0   model_1/model/global_average_pooling2d/Mean/re...                [2]   
1                                              Conv2D      [24  3  3  3]   
2                                               Add/y               [24]   
3                                           depthwise      [ 1  3  3 24]   
4                                             Add_1/y               [24]   
5                                            Conv2D_1      [16  1  1 24]   
6   model_1/model/batch_normalization_2/FusedBatch...               [16]   
7                                            Conv2D_2      [64  1  1 16]   
8                                             Add_2/y               [64]   
9                                         depthwise_1      [ 1  3  3 64]   
10                                            Add_3/y               [64]   
11                                           Conv2D_3      [24  1

#### Memory Analysis

In [54]:
print_section("MEMORY ANALYSIS")

# Runtime memory estimation
total_tensor_memory = tensor_df["MemoryBytes"].sum()

print(f"Serialized Model Size          : {bytes_to_human_readable(model_size)}")

print(
    f"Estimated Runtime Tensor Memory: "
    f"{bytes_to_human_readable(total_tensor_memory)}"
)

runtime_ratio = total_tensor_memory / model_size

print(f"Runtime Expansion Ratio        : {runtime_ratio:.2f}x")

print("\nScientific Commentary:")
print("""
TensorFlow Lite runtime memory is typically much larger than the
serialized .tflite file because intermediate activations,
temporary buffers, and operator workspaces are allocated during
inference.

Therefore:

    runtime_tensor_memory != serialized_model_size

Large runtime expansion ratios are common for:
- CNNs
- feature pyramid networks
- object detectors
- MediaPipe models
""")



MEMORY ANALYSIS
Serialized Model Size          : 3.12 MB
Estimated Runtime Tensor Memory: 8.82 MB
Runtime Expansion Ratio        : 2.82x

Scientific Commentary:

TensorFlow Lite runtime memory is typically much larger than the
serialized .tflite file because intermediate activations,
temporary buffers, and operator workspaces are allocated during
inference.

Therefore:

    runtime_tensor_memory != serialized_model_size

Large runtime expansion ratios are common for:
- CNNs
- feature pyramid networks
- object detectors
- MediaPipe models



#### Operator Analysis

In [55]:
print_section("OPERATOR ANALYSIS")

ops = []

for tensor in tensor_details:

    name = tensor["name"]

    if "/" in name:
        op_name = name.split("/")[0]
    else:
        op_name = name

    ops.append(op_name)

ops_series = pd.Series(ops)
op_counts = ops_series.value_counts()

print(op_counts.head(20))


OPERATOR ANALYSIS
model_1             79
Identity_dense       2
Identity_3_dense     2
Add_19               1
depthwise_11         1
Conv2D_26            1
Conv2D_25            1
Add_25               1
depthwise_12         1
Add_24               1
Conv2D_24            1
Conv2D_23            1
Add_23               1
Add_22               1
Conv2D_19            1
depthwise_13         1
depthwise_9          1
Conv2D_22            1
Conv2D_21            1
Add_21               1
Name: count, dtype: int64


#### Model Benchmark

In [56]:
print_section("INFERENCE BENCHMARK")

import time

# Generate dummy input
dummy_inputs = []

for inp in input_details:

    shape = inp["shape"]
    dtype = inp["dtype"]

    dummy = np.random.random(shape).astype(dtype)

    dummy_inputs.append(dummy)

# Warm-up
for idx, inp in enumerate(input_details):
    interpreter.set_tensor(inp["index"], dummy_inputs[idx])

interpreter.invoke()

# Benchmark
runs = 100

start = time.time()

for _ in range(runs):

    for idx, inp in enumerate(input_details):
        interpreter.set_tensor(inp["index"], dummy_inputs[idx])

    interpreter.invoke()

end = time.time()

avg_time = ((end - start) / runs) * 1000

print(f"Average Inference Time in {runs} runs: {avg_time:.3f} ms")


INFERENCE BENCHMARK
Average Inference Time in 100 runs: 4.284 ms


#### Quantization Parameter Inspection

In [57]:
print_section("QUANTIZATION PARAMETER INSPECTION")

quantized_tensor_count = 0

for tensor in tensor_details:

    qparams = tensor["quantization_parameters"]

    scales = qparams.get("scales", [])

    if len(scales) > 0:

        quantized_tensor_count += 1

        print(f"\nTensor: {tensor['name']}")
        print(f"Scales     : {scales[:5]}")
        print(f"Zero Points: {qparams.get('zero_points', [])[:5]}")

print(f"\nTotal Quantized Tensors: {quantized_tensor_count}")


QUANTIZATION PARAMETER INSPECTION

Tensor: Conv2D
Scales     : [0.07160433 0.00026817 0.00383512 0.0972564  0.00901206]
Zero Points: [0 0 0 0 0]

Tensor: Add/y
Scales     : [2.8080132e-04 1.0516479e-06 1.5039696e-05 3.8139767e-04 3.5341403e-05]
Zero Points: [0 0 0 0 0]

Tensor: depthwise
Scales     : [0.00162344 0.6072835  0.06050074 0.00167054 0.02414493]
Zero Points: [0 0 0 0 0]

Tensor: Add_1/y
Scales     : [3.8198566e-05 1.4289023e-02 1.4235468e-03 3.9306757e-05 5.6811603e-04]
Zero Points: [0 0 0 0 0]

Tensor: Conv2D_1
Scales     : [0.00489435 0.00668599 0.010004   0.00644377 0.0079586 ]
Zero Points: [0 0 0 0 0]

Tensor: model_1/model/batch_normalization_2/FusedBatchNormV3_model_1/model/conv2d_1/Conv2D1;Conv2D_1;model_1/model/batch_normalization_2/FusedBatchNormV3_model_1/model/conv2d_1/Conv2D1/y
Scales     : [0.00011516 0.00015732 0.00023539 0.00015162 0.00018726]
Zero Points: [0 0 0 0 0]

Tensor: Conv2D_2
Scales     : [3.1911295e-03 1.0000000e-06 2.1222932e-03 2.4129553e-02 1.00

#### Conclusion

In [58]:
print_section("CONCLUSION")

print(f"""
Model Summary
-------------------------------------------------------------------------------
Model File                : {MODEL_PATH}
Model Size                : {bytes_to_human_readable(model_size)}
Quantization Type         : {quantization_type}
Input Tensor Count        : {len(input_details)}
Output Tensor Count       : {len(output_details)}
Total Tensor Count        : {len(tensor_details)}
Average Inference Time    : {avg_time:.3f} ms

Key Findings
-------------------------------------------------------------------------------
1. Quantization analysis indicates:
   -> {quantization_type}

2. The model architecture suggests:
   -> {'Image-based CNN architecture' if len(input_details[0]['shape']) == 4 else 'Vector-based architecture'}

3. Memory footprint:
   -> Data Buffers consume the majority of storage.

4. Deployment suitability:
   -> {'Highly suitable for edge deployment' if 'INT8' in quantization_type else 'Better suited for high-precision environments'}

5. Scientific interpretation:
   -> Quantization trades numerical precision for improved
      memory efficiency and lower inference latency.
""")


CONCLUSION

Model Summary
-------------------------------------------------------------------------------
Model File                : hand_landmark_quant.tflite
Model Size                : 3.12 MB
Quantization Type         : INT8 Quantization
Input Tensor Count        : 1
Output Tensor Count       : 4
Total Tensor Count        : 172
Average Inference Time    : 4.284 ms

Key Findings
-------------------------------------------------------------------------------
1. Quantization analysis indicates:
   -> INT8 Quantization

2. The model architecture suggests:
   -> Image-based CNN architecture

3. Memory footprint:
   -> Data Buffers consume the majority of storage.

4. Deployment suitability:
   -> Highly suitable for edge deployment

5. Scientific interpretation:
   -> Quantization trades numerical precision for improved
      memory efficiency and lower inference latency.

